# Load Dependencies

In [ ]:
# Data analysis libraries
import numpy as np
import pandas as pd; pd.options.display.max_columns = 200
import geopandas as gpd
import linref as lr
import pyproj

# Visualization libraries
import plotly.express as px

# Utility libraries
import os

In [ ]:
# Define global variables
PROJECT_CRS = pyproj.CRS.from_user_input('EPSG:3857')

# Load Data

In [ ]:
# Point this to the location of the Geopackage file
fp = os.path.join('..', '99_Resources', 'franklin_county_training_data.gpkg')

# List all the layers in the file
gpd.list_layers(fp)

In [ ]:
# Load roadway data
# Includes all arterial and collector roadways in Franklin County, excluding freeways and interstates
query = """
SELECT NLF_ID, CTL_BEGIN_, CTL_END_NB, SEGMENT_LE, STREET_NAM, ADT_TOTAL_, geom
FROM roadways
"""
roadways = gpd.read_file(fp, sql=query) # Load a full layer as-is
roadways.to_crs(PROJECT_CRS, inplace=True)

# Load crash data
# Includes crashes from 2019-2023 in Franklin County of KABC severities, on the selected ODOT roadways
query = """
SELECT OBJECTID, CRASH_YR, KABCO, geom
FROM crashes_enriched
"""
crashes = gpd.read_file(fp, sql=query) # Load data using a query
crashes.to_crs(PROJECT_CRS, inplace=True)

print(f'Data loaded: {len(roadways):,.0f} roadways, {len(crashes):,.0f} crashes')

In [ ]:
# Apply categorical ordering to KABCO
crashes['KABCO'] = pd.Categorical(crashes['KABCO'], categories=['K', 'A', 'B', 'C'], ordered=True)

# Geometric Manipulation of Spatial Data

## Buffering

In [ ]:
# Create a buffer around the roadway data
buffer_distance = 100
roadways_buffered = roadways.set_geometry(roadways.buffer(100))

# Explore the buffered roadways
roadways_buffered.explore()

## Spatial Reductions

In [ ]:
# Prepare the unary union of the buffered roadways
union = roadways_buffered.dissolve()[['geometry']]
union.explore()

In [ ]:
# Prepare the convex hull of the buffered roadways
convex_hull = roadways_buffered.convex_hull.to_frame()
convex_hull.explore()

# Spatial Join
Documentation link: https://geopandas.org/en/stable/gallery/spatial_joins.html

## Joining Crashes to Roadways

In [ ]:
# Join all crashes which intersect with each buffered roadway
roadways_joined = roadways_buffered.sjoin(crashes, how='left', predicate='intersects')

# Take a look at the results
roadways_joined

In [ ]:
# This has produced a one-to-many join, where each roadway is duplicated for each crash that intersects with it.
# Let's use this to summarize crashes by street name and by crash severity.
severity_summary = roadways_joined.groupby(['STREET_NAM'])['KABCO'].value_counts().unstack().fillna(0).astype(int)
severity_summary

In [ ]:
# Let's compute a crash score for each roadway using a severity-weighted crash count
multiplier = pd.Series({'K': 25, 'A': 10, 'B': 5, 'C': 1})
# Apply the multiplier to the summary table and sum the results, selecting the top roads
score_summary = severity_summary.multiply(multiplier, axis=1).sum(axis=1)
score_summary.sort_values(ascending=False).head(10)

In [ ]:
# Let's compute an approximate annual MVMT for each roadway
# First, compute the MVMT per segment
roadways['VMT'] = roadways['ADT_TOTAL_'] * roadways['SEGMENT_LE'] * 365 / 1E6
# Now, compute the VMT per street name
vmt_summary = roadways.groupby(['STREET_NAM'])['VMT'].sum().fillna(0).round(2)
vmt_summary

In [ ]:
# Create a scatterplot of the crash score vs. the annual MVMT
fig = px.scatter(
    x=vmt_summary,
    y=score_summary,
    labels={'x': 'Annual MVMT', 'y': 'Crash Score'},
    trendline='ols',
    width=1000,
    height=600,
    template='plotly_white',
)

# Additional formatting
fig.update_xaxes(range=[0, 50])
fig.update_yaxes(range=[0, 4000])

# Show the plot
fig.show()

## Joining Roadways to Crashes

In [ ]:
# Join the nearest roadway to each crash
crashes_joined = crashes.sjoin_nearest(roadways, max_distance=100, how='left')

# Interestingly, this can sometimes produce duplicate rows, due to equal distances to multiple roadways.
# To address this, let's drop duplicates based on the OBJECTID of the crash.
crashes_joined.drop_duplicates(subset=['OBJECTID'], inplace=True)

# Take a look at the results
crashes_joined

In [ ]:
# Let's take a look at only the crashes that are within 100 feet of High Street using this joined information
crashes_joined[crashes_joined['STREET_NAM'] == 'HIGH'].explore(column='KABCO', cmap='RdYlGn')